# Cross-arm comparison — visualization

This notebook is the one place that compares *across* stages -- baseline (Qwen
and Claude), RAG, and fine-tuning -- rather than living inside any single
stage's own notebook. `mongodb_nl_to_sql_1.ipynb` owns baseline-only charts,
`rag/mongodb_nl_to_sql_rag.ipynb` owns RAG-only charts, and
`fine_tuning/visualize_results.ipynb` owns fine-tuning-only charts -- this one
is for anything that needs data from more than one of those stages at once, so
no single stage's notebook has to reach outside its own folder.

All charts are saved to `outputs/figures/` (the same figures folder the
baseline notebook already uses) since these are project-level, not
stage-level, outputs.

Like the other visualization notebooks, this one only reads already-computed
result files -- it doesn't generate or score anything itself.

In [1]:
import json
from collections import Counter, defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

ROOT = Path.cwd()
for _ in range(3):
    if (ROOT / "data").exists() and (ROOT / "rag").exists() and (ROOT / "fine_tuning").exists():
        break
    ROOT = ROOT.parent

FIGURES = ROOT / "outputs" / "figures"
FIGURES.mkdir(parents=True, exist_ok=True)

COLORS = ["#2eb872", "#f0a500", "#c98a2c", "#c0392b"]  # correct, wrong-nonempty, empty, hard-fail
STAGE_LABELS = ["Correct\n(exec. accuracy)", "Ran, non-empty\nbut wrong", "Ran, empty\nresult", "Hard failure\n(syntax/safety/error)"]

In [2]:
# ---------------------------------------------------
# Load every result file this notebook needs, and fail loud if any are
# missing -- these are all produced elsewhere in the pipeline
# (execute_queries.py / score_rag.py / the fine-tuning generation script).
# ---------------------------------------------------
paths = {
    "claude_305": ROOT / "data" / "claude_execution_results.json",
    "qwen_305": ROOT / "data" / "qwen_execution_results.json",
    "rag_61": ROOT / "rag" / "data" / "qwen_rag_execution_results.json",
    "baseline_61": ROOT / "rag" / "data" / "qwen_baseline_testslice_execution_results.json",
    "ft_61": ROOT / "data" / "finetuned_execution_results.json",
    "holdout": ROOT / "fine_tuning" / "data" / "holdout_eval_cases.json",
}
for name, p in paths.items():
    if not p.exists():
        raise FileNotFoundError(f"{p} missing -- run the {name} pipeline stage first")

data = {name: json.load(open(p)) for name, p in paths.items()}

holdout_ids = {h["id"] for h in data["holdout"]}
data["claude_61"] = [r for r in data["claude_305"] if r["id"] in holdout_ids]
assert len(data["claude_61"]) == 61, "Claude matched-61 slice did not come out to 61 cases -- check holdout ids"


def breakdown(records):
    total = len(records)
    correct = sum(1 for r in records if r.get("execution_accuracy"))
    hard_fail = sum(1 for r in records if r.get("status") != "PASS")
    wrong_nonempty = sum(1 for r in records if r.get("status") == "PASS" and r.get("non_empty_rate") and not r.get("execution_accuracy"))
    empty_but_ran = sum(1 for r in records if r.get("status") == "PASS" and not r.get("non_empty_rate"))
    assert correct + wrong_nonempty + empty_but_ran + hard_fail == total, "breakdown doesn't sum to total"
    return {"total": total, "correct": correct, "wrong_nonempty": wrong_nonempty, "empty": empty_but_ran, "hard_fail": hard_fail}


b = {
    "claude_305": breakdown(data["claude_305"]),
    "qwen_305": breakdown(data["qwen_305"]),
    "rag_61": breakdown(data["rag_61"]),
    "baseline_61": breakdown(data["baseline_61"]),
    "ft_61": breakdown(data["ft_61"]),
    "claude_61": breakdown(data["claude_61"]),
}
for k, v in b.items():
    print(k, v)

claude_305 {'total': 305, 'correct': 168, 'wrong_nonempty': 110, 'empty': 8, 'hard_fail': 19}
qwen_305 {'total': 305, 'correct': 13, 'wrong_nonempty': 84, 'empty': 173, 'hard_fail': 35}
rag_61 {'total': 61, 'correct': 15, 'wrong_nonempty': 27, 'empty': 11, 'hard_fail': 8}
baseline_61 {'total': 61, 'correct': 3, 'wrong_nonempty': 17, 'empty': 34, 'hard_fail': 7}
ft_61 {'total': 61, 'correct': 20, 'wrong_nonempty': 27, 'empty': 11, 'hard_fail': 3}
claude_61 {'total': 61, 'correct': 28, 'wrong_nonempty': 28, 'empty': 1, 'hard_fail': 4}


**Note on Claude's full-305 correct count**: this notebook recomputes `168/305`
(55.1%) directly from `data/claude_execution_results.json` as it stands today.
Earlier project documentation recorded `169/305` (55.4%) for this same figure --
a 1-case discrepancy. This does not change any conclusion in the report, but if
the exact number matters for the final submission, re-verify against the current
file rather than the previously-recorded figure (the file may have been
regenerated at some point after that number was first written down).

In [3]:
def outcome_chart(bd, title, fname):
    n = bd["total"]
    vals = [bd["correct"], bd["wrong_nonempty"], bd["empty"], bd["hard_fail"]]
    plt.figure(figsize=(6.5, 4))
    bars = plt.bar(STAGE_LABELS, vals, color=COLORS)
    plt.ylabel(f"Number of cases (of {n})")
    plt.title(title)
    plt.ylim(0, max(vals) + max(5, int(max(vals) * 0.2)))
    for bar, c in zip(bars, vals):
        plt.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + max(vals) * 0.02,
                  f"{c}/{n}\n({c/n*100:.1f}%)", ha="center", va="bottom", fontsize=9)
    plt.tight_layout()
    plt.savefig(FIGURES / fname, dpi=300)
    plt.close()
    print(f"Saved {FIGURES / fname}")


# Figure: RAG (K=10) outcome breakdown, 61-case holdout
outcome_chart(b["rag_61"], "RAG (K=10) Outcome Breakdown, 61-case Holdout", "rag_outcome_breakdown.png")

# Figure: Qwen baseline outcome breakdown, full 305-case set (Stage 1 headline)
outcome_chart(b["qwen_305"], "Qwen2.5-Coder Baseline Outcome Breakdown, Full 305-case Set", "baseline_qwen_outcome_breakdown.png")

# Figure: Claude baseline outcome breakdown, full 305-case set (Stage 1 headline)
outcome_chart(b["claude_305"], "Claude Baseline Outcome Breakdown, Full 305-case Set", "baseline_claude_outcome_breakdown.png")

Saved /tmp/chartdev/mock/outputs/figures/rag_outcome_breakdown.png
Saved /tmp/chartdev/mock/outputs/figures/baseline_qwen_outcome_breakdown.png


Saved /tmp/chartdev/mock/outputs/figures/baseline_claude_outcome_breakdown.png


In [4]:
# ---------------------------------------------------
# Figure: outcome composition across all four arms, on the identical matched
# 61-case holdout -- the single most useful "big picture" comparison in the
# project, since it puts every arm's failure mix on one common footing.
# ---------------------------------------------------
arms = ["baseline_61", "rag_61", "ft_61", "claude_61"]
arm_labels = ["Qwen\nBaseline", "Qwen\n+ RAG (K=10)", "Qwen\n+ LoRA Fine-tuned", "Claude\n(reference)"]
categories = ["correct", "wrong_nonempty", "empty", "hard_fail"]
cat_labels = ["Correct", "Ran, wrong", "Ran, empty", "Hard failure"]

plt.figure(figsize=(8, 5))
bottoms = np.zeros(len(arms))
for cat, color, clabel in zip(categories, COLORS, cat_labels):
    counts = np.array([b[a][cat] for a in arms])
    pct = counts / 61 * 100
    bars = plt.bar(arm_labels, pct, bottom=bottoms, color=color, label=clabel)
    for bar, c in zip(bars, counts):
        if c > 0:
            plt.text(bar.get_x() + bar.get_width() / 2, bar.get_y() + bar.get_height() / 2,
                      f"{c}", ha="center", va="center", fontsize=9,
                      color="white" if clabel in ("Correct", "Hard failure") else "black")
    bottoms += pct
plt.ylabel("Share of 61 held-out cases (%)")
plt.title("Outcome Composition Across Arms (matched 61-case holdout)")
plt.ylim(0, 100)
plt.legend(loc="upper center", bbox_to_anchor=(0.5, -0.12), ncol=4, fontsize=9, frameon=False)
plt.tight_layout()
plt.savefig(FIGURES / "all_arms_outcome_composition.png", dpi=300)
plt.close()
print(f"Saved {FIGURES / 'all_arms_outcome_composition.png'}")

Saved /tmp/chartdev/mock/outputs/figures/all_arms_outcome_composition.png


In [5]:
# ---------------------------------------------------
# Figure: execution accuracy stratified by question complexity.
# The dataset's complexity field has a known labeling inconsistency (28 of
# the original 121 references use "high"/"complex" instead of the standard
# "easy"/"medium"/"hard" -- documented in the project's own status notes) --
# both are folded into "hard" here since they were never meant as a 5th tier.
# ---------------------------------------------------
def norm_complexity(c):
    return "hard" if c in ("hard", "high", "complex") else (c or "unknown")


id_to_complexity = {h["id"]: norm_complexity(h.get("complexity")) for h in data["holdout"]}


def accuracy_by_complexity(records):
    buckets = defaultdict(lambda: [0, 0])
    for r in records:
        c = id_to_complexity.get(r["id"], "unknown")
        buckets[c][1] += 1
        if r.get("execution_accuracy"):
            buckets[c][0] += 1
    return {c: {"correct": buckets[c][0], "total": buckets[c][1],
                "pct": 100 * buckets[c][0] / buckets[c][1] if buckets[c][1] else 0}
            for c in ["easy", "medium", "hard"]}


cx = {
    "baseline": accuracy_by_complexity(data["baseline_61"]),
    "rag": accuracy_by_complexity(data["rag_61"]),
    "finetuned": accuracy_by_complexity(data["ft_61"]),
    "claude": accuracy_by_complexity(data["claude_61"]),
}

arm_keys = ["baseline", "rag", "finetuned", "claude"]
arm_names = ["Qwen Baseline", "Qwen + RAG (K=10)", "Qwen + Fine-tuned", "Claude (reference)"]
arm_colors = ["#888888", "#2a7de1", "#2eb872", "#7A2E2E"]
complexities = ["easy", "medium", "hard"]
x = np.arange(len(complexities))
width = 0.2

plt.figure(figsize=(7.5, 4.5))
for i, (ak, an, ac) in enumerate(zip(arm_keys, arm_names, arm_colors)):
    vals = [cx[ak][c]["pct"] for c in complexities]
    counts = [(cx[ak][c]["correct"], cx[ak][c]["total"]) for c in complexities]
    offset = (i - 1.5) * width
    bars = plt.bar(x + offset, vals, width, label=an, color=ac)
    for bar, v, (co, to) in zip(bars, vals, counts):
        plt.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1.5,
                  f"{co}/{to}", ha="center", va="bottom", fontsize=7)
n_by_c = {c: cx["baseline"][c]["total"] for c in complexities}
plt.xticks(x, [f"Easy\n(n={n_by_c['easy']})", f"Medium\n(n={n_by_c['medium']})", f"Hard\n(n={n_by_c['hard']})"])
plt.ylabel("Execution Accuracy (%)")
plt.title("Execution Accuracy by Question Complexity (matched 61-case holdout)")
plt.ylim(0, 65)
plt.legend(fontsize=8, loc="upper left")
plt.tight_layout()
plt.savefig(FIGURES / "accuracy_by_complexity.png", dpi=300)
plt.close()
print(f"Saved {FIGURES / 'accuracy_by_complexity.png'}")

print("\nFine-tuned accuracy by complexity:", cx["finetuned"])
print("RAG accuracy by complexity:       ", cx["rag"])

Saved /tmp/chartdev/mock/outputs/figures/accuracy_by_complexity.png

Fine-tuned accuracy by complexity: {'easy': {'correct': 7, 'total': 21, 'pct': 33.333333333333336}, 'medium': {'correct': 13, 'total': 28, 'pct': 46.42857142857143}, 'hard': {'correct': 0, 'total': 12, 'pct': 0.0}}
RAG accuracy by complexity:        {'easy': {'correct': 6, 'total': 21, 'pct': 28.571428571428573}, 'medium': {'correct': 7, 'total': 28, 'pct': 25.0}, 'hard': {'correct': 2, 'total': 12, 'pct': 16.666666666666668}}


### Reading this last chart

The complexity breakdown surfaces something the single headline accuracy
numbers hide: fine-tuning's overall lead over RAG comes entirely from *medium*-
complexity cases (13/28, 46.4% vs RAG's 7/28, 25.0%) -- on *hard* cases,
fine-tuning scores **0/12**, while RAG still gets 2/12 right. So "fine-tuning
beats RAG" is true in aggregate but not uniformly true -- RAG's retrieved
few-shot examples appear to give it some purchase on hard cases that the
fine-tuned adapter's baked-in behavior doesn't generalize to. This is a
concrete, testable angle for the still-open "why did fine-tuning beat RAG"
investigation noted elsewhere in the project.

# Where the accuracy actually comes from -- case-level overlap and a failure-mode taxonomy

Two more views, both computed on the identical matched 61-case holdout used above:

1. **Correct-set overlap between RAG and fine-tuned.** The headline numbers (15/61 vs
   20/61) only say fine-tuned wins in aggregate -- they say nothing about whether the
   two arms are getting *the same* cases right or genuinely different ones. If they
   diverge a lot, that's direct evidence for an ensemble/routing strategy being worth
   trying; if they mostly overlap, fine-tuning has just strictly subsumed RAG's wins.
2. **A failure-mode taxonomy**, classifying every non-correct case (across all four
   matched-61 arms) into a small set of mechanically-detected buckets: rejected before
   execution by the safety checker, a known invalid-operator misuse ($size/$min/$max
   used where MongoDB doesn't allow it), a runtime database error, a missing `$lookup`
   the gold query has but the generated one doesn't, a missing numeric type-cast the
   gold query has but the generated one doesn't, or an unclassified "other wrong logic"
   catch-all.

**On rigor**: every bucket here is assigned by a deterministic, checkable rule -- the
literal error message text for FAIL cases, or a direct comparison against the gold
`normalized_query` for PASS-but-wrong cases (does gold use `$lookup`/`$toInt`/`$toDouble`/
`$convert` where the generated query doesn't). Nothing here is inferred by reading
natural language or guessing intent. That means "other wrong logic" is deliberately a
large, honest catch-all rather than a false-precision breakdown -- those cases have
genuine wrong filters, wrong fields, or wrong aggregation logic that would need an
actual manual read to sub-classify further, which this notebook doesn't claim to do.

In [6]:
# ---------------------------------------------------
# Setup: gold reference queries (for the join/cast-detection heuristics) --
# fail loud if missing, same convention as every other data source above.
# ---------------------------------------------------
import re

gold_path = ROOT / "data" / "reference_queries.json"
if not gold_path.exists():
    raise FileNotFoundError(f"{gold_path} missing -- this is the 305-case golden dataset, should always exist")

gold = {r["id"]: r["normalized_query"] for r in json.load(open(gold_path))}


def classify_failure(rec):
    """Deterministic bucket for one non-correct case. Priority order matters:
    a FAIL case is classified purely from its own error text; a PASS-but-wrong
    case is classified by comparing its query text against the gold query for
    the same id (never by reading the natural-language question)."""
    if rec.get("execution_accuracy"):
        return "correct"

    if rec.get("status") != "PASS":
        err = rec.get("error") or ""
        if "REJECTED by safety check" in err:
            return "blocked_pre_execution"
        if (re.search(r"\$(size|min|max)\b", err)
                or "unknown group operator" in err
                or "must be an array" in err.lower()
                or "expected a number" in err.lower()):
            return "invalid_operator_misuse"
        if "$convert" in err or "failed to parse number" in err.lower():
            return "missing_type_cast"
        return "runtime_db_error"

    # status == PASS but execution_accuracy is False (wrong_nonempty or empty)
    q = rec.get("query") or ""
    g = gold.get(rec["id"], "")
    if "$lookup" in g and "$lookup" not in q:
        return "missing_join"
    if any(op in g for op in ("$toInt", "$toDouble", "$convert")) and not any(op in q for op in ("$toInt", "$toDouble", "$convert")):
        return "missing_type_cast"
    return "other_wrong_logic"


TAXONOMY_CATS = ["correct", "blocked_pre_execution", "invalid_operator_misuse",
                  "runtime_db_error", "missing_join", "missing_type_cast", "other_wrong_logic"]
TAXONOMY_COLORS = {
    "correct": "#2eb872",
    "blocked_pre_execution": "#c0392b",
    "invalid_operator_misuse": "#f0a500",
    "runtime_db_error": "#e67e22",
    "missing_join": "#8e44ad",
    "missing_type_cast": "#2980b9",
    "other_wrong_logic": "#95a5a6",
}
TAXONOMY_LABELS = {
    "correct": "Correct",
    "blocked_pre_execution": "Blocked pre-execution\n(safety/syntax)",
    "invalid_operator_misuse": "Invalid operator\n($size/$min/$max)",
    "runtime_db_error": "Runtime DB error\n(other)",
    "missing_join": "Missing $lookup\n(gold has one, output doesn't)",
    "missing_type_cast": "Missing type cast\n(gold casts, output doesn't)",
    "other_wrong_logic": "Other wrong logic\n(unclassified)",
}

tax = {}
for arm in ["baseline_61", "rag_61", "ft_61", "claude_61"]:
    counts = Counter(classify_failure(r) for r in data[arm])
    assert sum(counts.values()) == 61, f"{arm}: taxonomy buckets don't sum to 61"
    tax[arm] = {c: counts.get(c, 0) for c in TAXONOMY_CATS}
    print(arm, tax[arm])

baseline_61 {'correct': 3, 'blocked_pre_execution': 3, 'invalid_operator_misuse': 0, 'runtime_db_error': 4, 'missing_join': 15, 'missing_type_cast': 5, 'other_wrong_logic': 31}
rag_61 {'correct': 15, 'blocked_pre_execution': 1, 'invalid_operator_misuse': 4, 'runtime_db_error': 1, 'missing_join': 7, 'missing_type_cast': 5, 'other_wrong_logic': 28}
ft_61 {'correct': 20, 'blocked_pre_execution': 2, 'invalid_operator_misuse': 1, 'runtime_db_error': 0, 'missing_join': 8, 'missing_type_cast': 7, 'other_wrong_logic': 23}
claude_61 {'correct': 28, 'blocked_pre_execution': 1, 'invalid_operator_misuse': 0, 'runtime_db_error': 0, 'missing_join': 1, 'missing_type_cast': 4, 'other_wrong_logic': 27}


In [7]:
# ---------------------------------------------------
# Figure: correct-set overlap between RAG and fine-tuned, on the matched
# 61-case holdout. Not a text-similarity or approximate match -- both sets
# are exact `id` sets where execution_accuracy is True.
# ---------------------------------------------------
rag_correct = {r["id"] for r in data["rag_61"] if r.get("execution_accuracy")}
ft_correct = {r["id"] for r in data["ft_61"] if r.get("execution_accuracy")}
claude_correct_61 = {r["id"] for r in data["claude_61"] if r.get("execution_accuracy")}
all_61_ids = {r["id"] for r in data["rag_61"]}

both = rag_correct & ft_correct
rag_only = rag_correct - ft_correct
ft_only = ft_correct - rag_correct
neither = all_61_ids - rag_correct - ft_correct
assert len(both) + len(rag_only) + len(ft_only) + len(neither) == 61

overlap_labels = ["Both correct", "RAG only", "Fine-tuned only", "Neither"]
overlap_counts = [len(both), len(rag_only), len(ft_only), len(neither)]
overlap_colors = ["#2eb872", "#2a7de1", "#7A2E2E", "#95a5a6"]

plt.figure(figsize=(6.5, 4))
bars = plt.bar(overlap_labels, overlap_counts, color=overlap_colors)
plt.ylabel("Number of cases (of 61)")
plt.title("Where RAG and Fine-tuned Diverge (61-case Holdout)")
plt.ylim(0, max(overlap_counts) + 6)
for bar, c in zip(bars, overlap_counts):
    plt.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1,
              f"{c}/61\n({c/61*100:.1f}%)", ha="center", va="bottom", fontsize=9)
plt.tight_layout()
plt.savefig(FIGURES / "rag_vs_finetuned_correct_overlap.png", dpi=300)
plt.close()
print(f"Saved {FIGURES / 'rag_vs_finetuned_correct_overlap.png'}")

union_correct = rag_correct | ft_correct
print(f"\nUnion of RAG-correct and fine-tuned-correct: {len(union_correct)}/61 ({len(union_correct)/61*100:.1f}%)"
      f" -- vs {len(rag_correct)}/61 for RAG alone and {len(ft_correct)}/61 for fine-tuned alone.")
print(f"Of the {len(neither)} cases neither RAG nor fine-tuned get right, Claude also misses "
      f"{len(neither - claude_correct_61)}/{len(neither)} -- these are the genuinely hardest cases in the holdout, not just weak spots for the small model.")

Saved /tmp/chartdev/mock/outputs/figures/rag_vs_finetuned_correct_overlap.png

Union of RAG-correct and fine-tuned-correct: 25/61 (41.0%) -- vs 15/61 for RAG alone and 20/61 for fine-tuned alone.
Of the 36 cases neither RAG nor fine-tuned get right, Claude also misses 26/36 -- these are the genuinely hardest cases in the holdout, not just weak spots for the small model.


In [8]:
# ---------------------------------------------------
# Figure: failure-mode taxonomy, stacked per arm, matched 61-case holdout.
# Bar height is always 61 (correct + every failure bucket), so arms are
# directly comparable -- same style as the outcome-composition chart above,
# just with the non-correct portion broken down further.
# ---------------------------------------------------
tax_arms = ["baseline_61", "rag_61", "ft_61", "claude_61"]
tax_arm_labels = ["Qwen\nBaseline", "Qwen\n+ RAG (K=10)", "Qwen\n+ LoRA Fine-tuned", "Claude\n(reference)"]

plt.figure(figsize=(8.5, 5.5))
bottoms = np.zeros(len(tax_arms))
for cat in TAXONOMY_CATS:
    counts = np.array([tax[a][cat] for a in tax_arms])
    bars = plt.bar(tax_arm_labels, counts, bottom=bottoms, color=TAXONOMY_COLORS[cat], label=TAXONOMY_LABELS[cat].split("\n")[0])
    for bar, c in zip(bars, counts):
        if c > 0:
            plt.text(bar.get_x() + bar.get_width() / 2, bar.get_y() + bar.get_height() / 2,
                      f"{c}", ha="center", va="center", fontsize=8,
                      color="white" if cat in ("correct", "blocked_pre_execution", "missing_join") else "black")
    bottoms += counts
plt.ylabel("Number of cases (of 61)")
plt.title("Failure-Mode Taxonomy Across Arms (matched 61-case holdout)")
plt.ylim(0, 65)
plt.legend(loc="upper center", bbox_to_anchor=(0.5, -0.15), ncol=3, fontsize=8, frameon=False)
plt.tight_layout()
plt.savefig(FIGURES / "bug_taxonomy_by_arm.png", dpi=300)
plt.close()
print(f"Saved {FIGURES / 'bug_taxonomy_by_arm.png'}")

print("\nMissing-join cases as a share of each arm's misses (61 - correct):")
for a, label in zip(tax_arms, tax_arm_labels):
    misses = 61 - tax[a]["correct"]
    mj = tax[a]["missing_join"]
    print(f"  {label.replace(chr(10), ' ')}: {mj}/{misses} misses are a missing $lookup ({mj/misses*100:.1f}%)" if misses else f"  {label}: no misses")

Saved /tmp/chartdev/mock/outputs/figures/bug_taxonomy_by_arm.png

Missing-join cases as a share of each arm's misses (61 - correct):
  Qwen Baseline: 15/58 misses are a missing $lookup (25.9%)
  Qwen + RAG (K=10): 7/46 misses are a missing $lookup (15.2%)
  Qwen + LoRA Fine-tuned: 8/41 misses are a missing $lookup (19.5%)
  Claude (reference): 1/33 misses are a missing $lookup (3.0%)


### Reading these two charts

**Overlap**: RAG and fine-tuned agree on 10 of their combined correct cases, but each
also gets cases the other misses (RAG: 5 unique; fine-tuned: 10 unique). Their union
covers 25/61 (41.0%) -- meaningfully more than either arm alone (15/61 or 20/61). That's
concrete, case-level evidence that the two arms fail differently, not just at different
*rates* -- worth citing directly if the improvement report considers an ensemble or a
router that picks RAG vs. fine-tuned per query. It also means "fine-tuning beats RAG"
somewhat understates the real opportunity: the ceiling of combining both is well above
either one's current number.

**Taxonomy**: the zero-shot baseline's single biggest identifiable failure mode is a
missing `$lookup` -- 15 of its 58 misses (`missing_join`), far more than RAG (7/46) or
fine-tuned (8/41). That's a specific, mechanical gap (not "the model is generally weak")
consistent with a small zero-shot model defaulting to single-collection queries when a
join is actually required -- and it's the exact gap both RAG (via retrieved examples)
and fine-tuning (via training data containing joins) partially close. The
`invalid_operator_misuse` bucket ($size/$min/$max misused) is small in every arm (0-4
cases) but appears in RAG, fine-tuned, *and* was independently documented for RAG's
earlier K=10 run in the project notes -- a small, recurring generation bug worth a
targeted fix (a stricter safety-check rule or a normalize.py-style post-hoc rewrite)
since it's cheap to fix and shows up across arms rather than being one-off noise.